# 9.2 TensorFlow Lite

本 notebook 使用合成三類表格分類資料訓練 Keras 模型，轉成 TensorFlow Lite 格式，並用 TFLite Interpreter 驗證推論結果。


## 1. 載入套件


In [ ]:
import json
import os
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(tf.__version__)


## 2. 建立資料

使用小型表格分類資料，讓範例聚焦在模型轉換與推論驗證。


In [ ]:
X, y = make_classification(
    n_samples=1800,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    n_classes=3,
    n_clusters_per_class=1,
    class_sep=1.25,
    flip_y=0.03,
    random_state=SEED,
)
X = X.astype('float32')
y = y.astype('int64')
class_names = ['class_a', 'class_b', 'class_c']

x_train_full, x_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_valid = scaler.transform(x_valid).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

print(x_train.shape, x_valid.shape, x_test.shape)


## 3. 訓練 Keras 模型


In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],), name='features'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(len(class_names), activation='softmax', name='probabilities'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_valid, y_valid),
    epochs=25,
    batch_size=32,
    verbose=0,
)

keras_loss, keras_acc = model.evaluate(x_test, y_test, verbose=0)
print({'keras_test_loss': keras_loss, 'keras_test_accuracy': keras_acc})


## 4. 儲存 Keras 模型與 Metadata


In [ ]:
artifact_dir = Path('artifacts/9_2_tflite')
artifact_dir.mkdir(parents=True, exist_ok=True)

keras_model_path = artifact_dir / 'classifier.keras'
model.save(keras_model_path)

metadata = {
    'class_names': class_names,
    'feature_count': int(x_train.shape[1]),
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
}
metadata_path = artifact_dir / 'metadata.json'
metadata_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print(keras_model_path)
print(metadata_path)


## 5. 轉換成 TensorFlow Lite


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = artifact_dir / 'classifier.tflite'
tflite_path.write_bytes(tflite_model)

converter_optimized = tf.lite.TFLiteConverter.from_keras_model(model)
converter_optimized.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_optimized_model = converter_optimized.convert()

tflite_optimized_path = artifact_dir / 'classifier_optimized.tflite'
tflite_optimized_path.write_bytes(tflite_optimized_model)

size_report = pd.DataFrame([
    {'file': tflite_path.name, 'size_kb': round(tflite_path.stat().st_size / 1024, 2)},
    {'file': tflite_optimized_path.name, 'size_kb': round(tflite_optimized_path.stat().st_size / 1024, 2)},
])
size_report


## 6. 使用 TFLite Interpreter 推論


In [ ]:
def predict_with_tflite(model_path, samples):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_index = input_details[0]['index']
    output_index = output_details[0]['index']
    predictions = []

    for sample in samples.astype('float32'):
        batch = np.expand_dims(sample, axis=0).astype(input_details[0]['dtype'])
        interpreter.set_tensor(input_index, batch)
        interpreter.invoke()
        pred = interpreter.get_tensor(output_index)[0]
        predictions.append(pred)

    return np.array(predictions)

sample_x = x_test[:20]
keras_sample_prob = model.predict(sample_x, verbose=0)
tflite_prob = predict_with_tflite(tflite_path, sample_x)
optimized_prob = predict_with_tflite(tflite_optimized_path, sample_x)

comparison = pd.DataFrame({
    'keras_class': keras_sample_prob.argmax(axis=1),
    'tflite_class': tflite_prob.argmax(axis=1),
    'optimized_class': optimized_prob.argmax(axis=1),
    'true_class': y_test[:20],
})
comparison


## 7. 比較 Keras 與 TFLite 結果


In [ ]:
keras_all_pred = model.predict(x_test, verbose=0).argmax(axis=1)
tflite_all_prob = predict_with_tflite(tflite_path, x_test)
tflite_pred = tflite_all_prob.argmax(axis=1)
optimized_all_prob = predict_with_tflite(tflite_optimized_path, x_test)
optimized_pred = optimized_all_prob.argmax(axis=1)

report = pd.DataFrame([
    {'model': 'keras', 'accuracy': accuracy_score(y_test, keras_all_pred)},
    {'model': 'tflite', 'accuracy': accuracy_score(y_test, tflite_pred)},
    {'model': 'tflite_optimized', 'accuracy': accuracy_score(y_test, optimized_pred)},
])
report


In [ ]:
print(classification_report(y_test, tflite_pred, target_names=class_names, digits=3))


## 8. 套用自己的資料

轉換成 TFLite 後一定要抽樣比對 Keras 與 TFLite 的輸出。若你的模型需要圖片 resize、文字向量化或表格標準化，部署端也要使用同一套前處理規格。
